# Day 6: Chunking Documents Using LangChain

Chunking our day 4 + 5 documents for our chatbot RAG- we start by loading in our files

In [2]:
import csv
import json
import random
import re
from datetime import datetime, timezone
from pathlib import Path

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

root = Path(r"C:\Users\micro\Documents\ABTalksAI-Cohort")
plans_path = root / "data" / "plans.csv"
raw_text_dir = root / "raw_text"

raw_text_files = sorted(raw_text_dir.glob("*.txt"))

with plans_path.open("r", encoding="utf-8", newline="") as handle:
    plans = list(csv.DictReader(handle))


In [3]:
ingested_at = datetime.now(timezone.utc).isoformat()
chunks = []

for plan_number, plan in enumerate(plans, start=1):
    plan_id = plan.get("id") or plan_number

    text_content = (
        f"Plan: {plan['plan_name']}. "
        f"Premium: ${plan['monthly_premium']}/month. "
        f"Deductible: ${plan['annual_deductible']}. "
        f"Coinsurance: {plan['copay_pct']}%. "
        f"Network: {plan['network_tier']}."
    )

    chunks.append(
        Document(
            page_content=text_content,
            metadata={
                "id": f"plan-{plan_id}",
                "text": text_content,
                "source_file": plans_path.name,
                "source_type": "structured",
                "plan_type": plan["plan_name"],
                "section": "coverage",
                "ingested_at": ingested_at,
            },
        )
    )
    
print(f"Created {len(chunks)} structured plan documents.")


Created 3 structured plan documents.


In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", "? ", "! ", " ", ""],
    chunk_size=500,
    chunk_overlap=50,
    keep_separator=True,
    strip_whitespace=True,
)

for raw_text_path in raw_text_files:
    text = raw_text_path.read_text(encoding="utf-8")

    text = text.replace("\r\n", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    section = raw_text_path.stem

    document = Document(
        page_content=text,
        metadata={
            "source_file": raw_text_path.name,
            "source_type": "unstructured",
            "plan_type": raw_text_path.stem,
            "section": section,
            "ingested_at": ingested_at,
        },
    )

    split_chunks = text_splitter.split_documents([document])

    for chunk_number, chunk in enumerate(split_chunks, start=1):
        chunk.metadata["id"] = (
            f"{raw_text_path.stem}-chunk-{chunk_number}"
        )
        chunk.metadata["text"] = chunk.page_content

    chunks.extend(split_chunks)


print(f"Created {len(chunks)} LangChain documents.")

Created 42 LangChain documents.


In [5]:
knowledge_base_path = root / "knowledge_base.jsonl"

with knowledge_base_path.open("w", encoding="utf-8") as handle:
    for chunk in chunks:
        record = {
            **chunk.metadata,
            "text": chunk.page_content,
        }
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Saved {len(chunks)} chunks to {knowledge_base_path}.")


Saved 42 chunks to C:\Users\micro\Documents\ABTalksAI-Cohort\knowledge_base.jsonl.


In [6]:
for chunk in random.sample(chunks, min(5, len(chunks))):
    print("=" * 80)
    print(chunk.page_content)
    print(chunk.metadata)


Plan: Gold PPO. Premium: $500/month. Deductible: $2000. Coinsurance: 10%. Network: Gold.
{'id': 'plan-1', 'text': 'Plan: Gold PPO. Premium: $500/month. Deductible: $2000. Coinsurance: 10%. Network: Gold.', 'source_file': 'plans.csv', 'source_type': 'structured', 'plan_type': 'Gold PPO', 'section': 'coverage', 'ingested_at': '2026-07-22T17:55:59.514035+00:00'}
Adjudicate: Approve, partially approve, pend, or deny each service line. Record clear reason codes.
Issue payment and explanation: Send payment to the correct payee and produce an explanation of benefits or remittance advice.
Handle adjustments and appeals: Correct processing errors, recover overpayments where appropriate, and conduct timely appeal reviews.
3. Intake Checklist
Claim received date is captured.
Member and patient identifiers match enrollment records.
{'source_file': 'claims_process.txt', 'source_type': 'unstructured', 'plan_type': 'claims_process', 'section': 'claims_process', 'ingested_at': '2026-07-22T17:55:59.514

## Result

42 total chunks. There were lots of messes and there are still blank chunks, but this is the overall best outcome I've seen when sampling randomly. I learned a lot about handling different data types and inserting metadata when using LangChain frameworks to develop a RAG pipeline.